Installing dependencies and importing libraries

In [1]:
!pip install transformers peft trl datasets scikit-learn accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.2 MB/s eta 0:00:00


In [2]:
!pip install torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.5 MB/s eta 0:00:00


In [3]:
import pandas as pd
import io as io
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split


In [ ]:
DATA SETUP

In [4]:
from google.colab import files
uploaded = files.upload()

Saving data.csv to data.csv


Checking for null and label counts

In [24]:
fn_data = pd.read_csv(io.BytesIO(uploaded['data.csv']))
print(fn_data.describe())
fn_data.isnull().sum()
print("\nLabel counts:\n", fn_data["Sentiment"].value_counts())


                                                 Sentence Sentiment
count                                                5842      5842
unique                                               5322         3
top     Operating loss totalled EUR 0.9 mn , down from...   neutral
freq                                                    2      3130

Label counts:
 Sentiment
neutral     3130
positive    1852
negative     860
Name: count, dtype: int64


Defining model and parameters for lora config

In [31]:
model_name = "gpt2"
lora_rank = 64
lora_alpha = 128
epochs = 10
lr = 2e-4
output_dir = "lora-output"
labels = ["positive", "negative", "neutral"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:",device)

using device: cuda


Chose To use stratify here because of imbalance in dataset for sentiment

In [7]:
train_df, test_df = train_test_split(
    fn_data, test_size=0.2, random_state=42, stratify=fn_data["Sentiment"]
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

def prepare(example):
  example["label_str"] = example["Sentiment"]
  example["text"] = (
      f"### financial News:\n{example["Sentence"]}\n\n"
      f"### Sentiment;\n{example['Sentiment']}"
  )
  return example

train_ds = train_ds.map(prepare)
test_ds = test_ds.map(prepare)
print(f"Train: {len(train_ds)}  Test: {len(test_ds)}")

Map:   0%|          | 0/4673 [00:00<?, ? examples/s]

Map:   0%|          | 0/1169 [00:00<?, ? examples/s]

Train: 4673  Test: 1169


Settting up tokenizer and lora config, testing trainable params

In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model= AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout = .01,
    target_modules = ["c_attn", "c_proj"],


)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 6,488,064 || all params: 130,927,872 || trainable%: 4.9554


Model training

In [33]:
trainer = SFTTrainer(
    model=model,
    args = SFTConfig(
        output_dir = output_dir,
        num_train_epochs = epochs,
        per_device_train_batch_size = 8,
        learning_rate = lr,
        fp16 = True,
        report_to = "none",
        dataset_text_field = "text"
    ),
    train_dataset = train_ds,
    processing_class = tokenizer,
)

trainer.train()
model.save_pretrained(f"{output_dir}/adapter")
print("Training done, Adapter Saved")

Adding EOS to train dataset:   0%|          | 0/4673 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4673 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
10,4.707767
20,3.687330
30,3.471344
40,3.077964
50,3.091628
60,3.238322
70,3.026685
80,2.937463
90,2.745621
100,2.867428


Training done, Adapter Saved


Model Evaluation

In [34]:
def predict(mdl, sentences):
  mdl.eval()
  preds=[]
  with torch.inference_mode():
    for sentence in sentences:
      prompt = (
          f"### financial news: :/n{sentence}\n\n"
          f"### Sentiment:\n"
      )
      inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to(device)
      out = mdl.generate(**inputs, max_new_tokens =5, do_sample = False, pad_token_id=tokenizer.eos_token_id)
      decoded = tokenizer.decode(out[0],skip_special_tokens=True).lower()
      pred = next((l for l in labels if l in decoded.split("### Sentiment:\n")[-1]), "neutral")
      preds.append(pred)
  return preds

sentences =test_ds["Sentence"]
golds=test_ds["Sentiment"]

base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base_preds = predict(base_model, sentences)


ft_model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(model_name), f"{output_dir}/adapter").to(device)

ft_preds = predict(ft_model, sentences)

for name, preds in [("Base (zero-shot)", base_preds), ("LoRA fine-tuned", ft_preds)]:
    print(f"\n── {name} ──")
    print(f"Accuracy: {accuracy_score(golds, preds):.3f}")
    print(f"Macro-F1: {f1_score(golds, preds, average='macro', labels=labels, zero_division=0):.3f}")
    print(classification_report(golds, preds, labels=labels, zero_division=0))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



── Base (zero-shot) ──
Accuracy: 0.540
Macro-F1: 0.253
              precision    recall  f1-score   support

    positive       0.75      0.01      0.02       371
    negative       0.57      0.02      0.04       172
     neutral       0.54      1.00      0.70       626

    accuracy                           0.54      1169
   macro avg       0.62      0.34      0.25      1169
weighted avg       0.61      0.54      0.39      1169


── LoRA fine-tuned ──
Accuracy: 0.787
Macro-F1: 0.726
              precision    recall  f1-score   support

    positive       0.83      0.84      0.83       371
    negative       0.50      0.51      0.51       172
     neutral       0.84      0.83      0.84       626

    accuracy                           0.79      1169
   macro avg       0.72      0.73      0.73      1169
weighted avg       0.79      0.79      0.79      1169

